# European Options Pricing and Portfolio Risk

This notebook demonstrates analytic pricing, Greeks, Monte Carlo convergence, implied volatility, scenario analysis, and performance measurement.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from time import perf_counter
import matplotlib.pyplot as plt
import numpy as np
from pricing.black_scholes import black_scholes_price
from pricing.implied_volatility import implied_volatility
from pricing.monte_carlo import MonteCarloSimulation
from risk.greeks import black_scholes_greeks
from risk.portfolio import OptionPosition, Portfolio
from visualization import (plot_greeks_vs_spot, plot_monte_carlo_convergence,
                           plot_option_price_vs_spot, plot_portfolio_pnl,
                           plot_volatility_smile)

## Black-Scholes Prices and Greeks

In [ ]:
spot, strike, volatility, rate, expiry = 100.0, 100.0, 0.20, 0.05, 1.0
call_price = black_scholes_price('call', spot, strike, volatility, rate, expiry)
put_price = black_scholes_price('put', spot, strike, volatility, rate, expiry)
call_greeks = black_scholes_greeks('call', spot, strike, volatility, rate, expiry)
call_price, put_price, call_greeks

In [ ]:
spots = np.linspace(60, 140, 161)
plot_option_price_vs_spot(spots, strike, volatility, rate, expiry)
plt.show()
plot_greeks_vs_spot(spots, 'call', strike, volatility, rate, expiry)
plt.tight_layout()
plt.show()

## Monte Carlo Pricing and Convergence

In [ ]:
simulation = MonteCarloSimulation(simulations=50_000, timesteps=252, seed=42)
payoffs = simulation.discounted_payoffs('call', spot, strike, volatility, rate, expiry)
checkpoints, estimates = simulation.convergence(payoffs)
result = simulation.price('call', spot, strike, volatility, rate, expiry)
result

In [ ]:
plot_monte_carlo_convergence(checkpoints, estimates, call_price)
plt.show()

## Synthetic Volatility Smile

In [ ]:
strikes = np.arange(80.0, 125.0, 5.0)
synthetic_vols = 0.18 + 0.00006 * (strikes - 102.5) ** 2
market_prices = [black_scholes_price('call', spot, k, vol, rate, expiry)
                 for k, vol in zip(strikes, synthetic_vols)]
recovered = [implied_volatility('call', price, spot, k, rate, expiry)
             for k, price in zip(strikes, market_prices)]
plot_volatility_smile(strikes, market_prices, 'call', spot, rate, expiry)
plt.show()
np.column_stack([strikes, synthetic_vols, recovered])

## Portfolio Risk and Scenario PnL

In [ ]:
portfolio = Portfolio([
    OptionPosition('call', 100.0, 1.0, 10.0),
    OptionPosition('put', 95.0, 0.5, -5.0),
    OptionPosition('call', 110.0, 1.5, 3.0),
])
portfolio.value(spot, volatility, rate), portfolio.aggregate_greeks(spot, volatility, rate)

In [ ]:
scenarios = portfolio.scenario_analysis(
    spot, volatility, rate,
    spot_changes=[-0.10, -0.05, 0.0, 0.05, 0.10],
    volatility_changes=[-0.05, 0.0, 0.05],
)
plot_portfolio_pnl(scenarios)
plt.show()
scenarios.head()

## Performance Comparison

In [ ]:
started = perf_counter()
_ = [black_scholes_price('call', spot, strike, volatility, rate, expiry) for _ in range(10_000)]
analytic_seconds = perf_counter() - started

started = perf_counter()
_ = simulation.price('call', spot, strike, volatility, rate, expiry)
monte_carlo_seconds = perf_counter() - started
{'10,000 Black-Scholes evaluations': analytic_seconds,
 'one 50,000-path Monte Carlo evaluation': monte_carlo_seconds}